In [1]:
import os

In [2]:
%pwd

'/home/ramon/Github/mlops-wine-prediction/research'

In [3]:
os.chdir("../")
%pwd

'/home/ramon/Github/mlops-wine-prediction'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.mlops_wine_prediction.constants import *
from src.mlops_wine_prediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config

In [9]:
import os
import urllib.request as request
from src.mlops_wine_prediction import logger
import zipfile

In [10]:
## Componente - Ingestão de Dados

class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    # Fazendo o download do arquivo zip
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} baixado! com as seguintes informações: \n{headers}")
        else:
            logger.info(f"O arquivo já existe")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extrai o arquivo zip para o diretório de dados
        A função não retorna nada
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [12]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-04-13 16:37:19,758: INFO: common: arquivo yaml: config/config.yaml carregado com sucesso]
[2026-04-13 16:37:19,762: INFO: common: arquivo yaml: params.yaml carregado com sucesso]
[2026-04-13 16:37:19,765: INFO: common: arquivo yaml: schema.yaml carregado com sucesso]
[2026-04-13 16:37:19,767: INFO: common: diretório criado em: artifacts]
[2026-04-13 16:37:19,772: INFO: common: diretório criado em: artifacts/data_ingestion]
[2026-04-13 16:37:20,763: INFO: 251335794: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: C626:330321:6BDBBE:D8336A:69DD45F0
Accept-Ranges: bytes
D